In [9]:
import os
import json
import re
# import pytesseract
import cv2
import kagglehub
import easyocr
from transformers import AutoProcessor, AutoModelForImageTextToText
import tqdm as notebook_tqdm

In [10]:
MODELS_DIR = "./models"

os.makedirs(MODELS_DIR, exist_ok=True)
print(f"DEBUG: Ensuring models directory exists at: {os.path.abspath(MODELS_DIR)}")

DEBUG: Ensuring models directory exists at: R:\Apps\DeepResearch\MedOCR\beta\models


In [11]:
os.environ['HF_HOME'] = MODELS_DIR
os.environ['HUGGINGFACE_HUB_CACHE'] = os.path.join(MODELS_DIR, 'hub')
os.environ['KAGGLEHUB_CACHE'] = os.path.join(MODELS_DIR, 'kagglehub')
print(f"DEBUG: HF_HOME set to: {os.environ.get('HF_HOME')}")
print(f"DEBUG: HUGGINGFACE_HUB_CACHE set to: {os.environ.get('HUGGINGFACE_HUB_CACHE')}")
print(f"DEBUG: KAGGLEHUB_CACHE set to: {os.environ.get('KAGGLEHUB_CACHE')}")

custom_kaggle_config_dir = os.path.dirname(os.path.abspath(__file__))
os.environ['KAGGLE_CONFIG_DIR'] = custom_kaggle_config_dir
print(f"DEBUG: KAGGLE_CONFIG_DIR set to: {os.environ.get('KAGGLE_CONFIG_DIR')}")

DEBUG: HF_HOME set to: ./models
DEBUG: HUGGINGFACE_HUB_CACHE set to: ./models\hub
DEBUG: KAGGLEHUB_CACHE set to: ./models\kagglehub


NameError: name '__file__' is not defined

In [ ]:
reader = easyocr.Reader(['en'])
result = reader.readtext('chinese.jpg')

In [8]:
def preprocess_image_for_ocr(image_path):

    try:
        img = cv2.imread(image_path)
        if img is None:
            print(
                f"DEBUG: Error: Could not load image from {image_path}. Check if file exists and is a valid image format.")
            return None

        gray_img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        print(f"DEBUG: Image converted to grayscale.")

        denoised_img = cv2.medianBlur(gray_img, 3)
        print(f"DEBUG: Image denoised with median blur.")

        preprocessed_img = cv2.adaptiveThreshold(
            denoised_img, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2
        )
        print(f"DEBUG: Image binarized with adaptive thresholding.")


        def determine_skew(image):
            # Calculate the skew angle of an image
            # This is a simplified example, a more robust solution might involve
            # Hough transforms or other methods.
            coords = np.column_stack(np.where(image > 0))
            angle = cv2.minAreaRect(coords)[-1]
            if angle < -45:
                angle = -(90 + angle)
            else:
                angle = -angle
            return angle

        def deskew_image(image, angle):
            # Rotate the image to correct the skew
            (h, w) = image.shape[:2]
            center = (w // 2, h // 2)
            M = cv2.getRotationMatrix2D(center, angle, 1.0)
            rotated = cv2.warpAffine(image, M, (w, h),
                flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)
            return rotated

        skew_angle = determine_skew(preprocessed_img)
        if abs(skew_angle) > 0.5:
            preprocessed_img = deskew_image(preprocessed_img, skew_angle)
            print(f"DEBUG: Image deskewed by {skew_angle:.2f} degrees.")

        print(f"DEBUG: Image preprocessed for OCR: {image_path}")
        return preprocessed_img
    except Exception as e:
        print(f"DEBUG: Error during image preprocessing for {image_path}: {e}")
        return None

In [ ]:
def correct_common_ocr_errors(text):
    corrected_text = text
    corrected_text = re.sub(r'(\d)(\d{2})\s*(U/L|mmol/L|mg/dL|g/dL)', r'\1.\2 \3', corrected_text)
    corrected_text = re.sub(r'MCHC:\s*(\d{2})0', r'MCHC: \1.0', corrected_text)
    print(f"DEBUG: Applied rule-based OCR corrections. Original length: {len(text)}, Corrected length: {len(corrected_text)}")
    return corrected_text

In [ ]:

processor, model = None, None
FINE_TUNED_MODEL_PATH = "./fine_tuned_gemma_medical_extractor"


try:
    if os.path.exists(FINE_TUNED_MODEL_PATH):
        print(f"DEBUG: Loading fine-tuned Gemma model from {FINE_TUNED_MODEL_PATH}")
        processor = AutoProcessor.from_pretrained(FINE_TUNED_MODEL_PATH)
        model = AutoModelForImageTextToText.from_pretrained(FINE_TUNED_MODEL_PATH, torch_dtype="auto", device_map="auto")
        print("DEBUG: Fine-tuned Gemma model loaded successfully.")
    else:
        print("DEBUG: Fine-tuned model not found. Attempting to load original Gemma 3n from KaggleHub.")
        GEMMA_PATH = kagglehub.model_download("google/gemma-3n/transformers/gemma-3n-e2b-it")

        processor = AutoProcessor.from_pretrained(GEMMA_PATH)
        model = AutoModelForImageTextToText.from_pretrained(GEMMA_PATH, torch_dtype="auto", device_map="auto")
        print(f"DEBUG: Gemma 3n model loaded successfully from {GEMMA_PATH}.")

except Exception as e:
    print(f"ERROR: Could not load Gemma model. Error: {e}")
    processor, model = None, None

In [ ]:

def extract_json_with_gemma(text_to_analyze,
                            query_text="Given the following medical lab report text, extract all relevant information into a JSON object. Use standardized medical key names."):
    if processor is None or model is None:
        print("Gemma model not loaded. Skipping Gemma extraction.")
        return None

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": query_text + "\n\nText: " + text_to_analyze}
            ]
        }
    ]

    try:
        inputs = processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt"
        ).to(model.device, dtype=model.dtype)
        input_len = inputs["input_ids"].shape[-1]

        outputs = model.generate(**inputs, max_new_tokens=1024,
                                 disable_compile=True)
        gemma_response_text = processor.batch_decode(
            outputs[:, input_len:],
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True
        )

        json_match = re.search(r'```json\n(.*?)\n```', gemma_response_text[0], re.DOTALL)
        if json_match:
            json_string = json_match.group(1)
        else:
            json_string = gemma_response_text[0].strip()

        try:
            extracted_json = json.loads(json_string)
            print("DEBUG: Successfully parsed Gemma's output as JSON.")
            return extracted_json
        except json.JSONDecodeError as e:
            print(f"DEBUG: Gemma output is not valid JSON. Attempting direct parsing or returning raw text. Error: {e}")
            print(f"DEBUG: Gemma raw output: {json_string}")
            return {"gemma_raw_output": json_string}

    except Exception as e:
        print(f"ERROR: During Gemma text generation or parsing: {e}")
        return None